In [1]:
from bluebikes_analysis.config import LOCAL_DATA_DIR, NCEI_APIKEY

In [12]:
NCEI_DATASET = "GHCND"
NCEI_STATION = f"USW00014739"

The weather data is sourced from NOAA's Global Historical Climatology Network Daily (GHCND) dataset, recorded at Boston Logan International Airport (`USW00014739`). Six daily variables are retrieved, all in standard units.

`TMAX` and `TMIN` are the maximum and minimum air temperatures of the day, reported in degrees Fahrenheit (°F). Together they capture the thermal range experienced by riders and are among the strongest predictors of bike-share demand.

`PRCP` is the total precipitation for the day in inches (in), combining rain and melted snow. `SNOW` records the fresh snowfall in inches, while `SNWD` measures the snow depth already on the ground — the distinction matters because even a dry day can suppress ridership if there is accumulated snow making roads and bike lanes hazardous.

`AWND` is the average wind speed for the day in miles per hour (mph). High winds are a meaningful deterrent to cycling and add explanatory power beyond temperature and precipitation alone.

In [ ]:
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta
import requests
import pandas as pd

BASE = "https://www.ncei.noaa.gov/cdo-web/api/v2"
HEADERS = {"token": NCEI_APIKEY}

DATATYPES = ["TMAX", "TMIN", "PRCP", "SNOW", "SNWD", "AWND"]


def _fetch_range(dataset, station, start, end):
    """Fetch all records between two dates, paginating as needed."""
    params = {
        "datasetid": dataset,
        "stationid": f"{dataset}:{station}",
        "datatypeid": ",".join(DATATYPES),
        "startdate": str(start),
        "enddate": str(end),
        "units": "standard",
        "limit": 1000,
        "includemetadata": "false",
    }
    records, offset = [], 1
    while True:
        params["offset"] = offset
        r = requests.get(f"{BASE}/data", headers=HEADERS, params=params)
        r.raise_for_status()
        batch = r.json().get("results", [])
        if not batch:
            break
        records.extend(batch)
        offset += 1000
    return records


def fetch_weather(dataset, station, start_date, end_date):
    """Fetch daily weather data and return a wide DataFrame indexed by date.

    Args:
        dataset:    NCEI dataset id (e.g. "GHCND").
        station:    Station id without dataset prefix (e.g. "USW00014739").
        start_date: Start date as "YYYY-MM-DD" string.
        end_date:   End date as "YYYY-MM-DD" string. Pass "yesterday" to use yesterday's date.
    """
    start = date.fromisoformat(start_date)
    end = date.today() - timedelta(days=1) if end_date == "yesterday" else date.fromisoformat(end_date)

    records = []
    cursor = start
    while cursor <= end:
        year_end = min(date(cursor.year, 12, 31), end)
        print(f"Fetching {cursor.year}...")
        batch = _fetch_range(dataset, station, cursor, year_end)
        print(f"  → {len(batch)} records")
        records.extend(batch)
        cursor = date(cursor.year + 1, 1, 1)

    if not records:
        raise ValueError("No records returned — check dataset, station, and API token.")

    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["date"]).dt.date
    return df.pivot_table(index="date", columns="datatype", values="value", aggfunc="first")


weather_df = fetch_weather(NCEI_DATASET, NCEI_STATION, start_date="2018-01-01", end_date="yesterday")
print(weather_df.head())

Fetching 2018...
  → 1825 records
Fetching 2019...
  → 1825 records
Fetching 2020...
  → 1830 records
Fetching 2021...
  → 1825 records
Fetching 2022...
  → 1825 records
Fetching 2023...
  → 1825 records
Fetching 2024...
  → 1830 records
Fetching 2025...
  → 1825 records
Fetching 2026...
  → 309 records
datatype    AWND  PRCP  SNOW  TMAX  TMIN
date                                    
2018-01-01  16.3  0.00   0.0  13.0   0.0
2018-01-02  12.8  0.00   0.0  19.0   4.0
2018-01-03   9.4  0.00   0.0  29.0  16.0
2018-01-04  22.6  1.35  13.4  30.0  22.0
2018-01-05  24.8  0.00   0.0  24.0   6.0


In [21]:
weather_df.columns.name = None

In [26]:
weather_df = weather_df.reset_index()